# Checkpoint 2 — Sixth Version

Pipeline identica alla quinta versione con tre micro-modifiche per migliorare punteggio ed efficienza:

1. **Sobol 2^14 invece di 2^15** al livello 1: dimezza il tempo del global search senza perdita di copertura significativa.
2. **Top-5 con distanziamento minimo (≥ 0.35)**: i seeds del multi-start L2 coprono bacini diversi invece di essere tutti vicini al minimo globale del surrogate.
3. **Fallback L3 → L2**: se il livello 3 (FMM 61×61) produce un costo peggiore del livello 2 ricomputato sulla stessa griglia, si tiene la soluzione L2. Evita il drift osservato in alcuni scenari.

Tutto il resto (FMM Numba, surrogate analitico, griglie, CP1, struttura della pipeline) è invariato rispetto alla quinta versione.


In [1]:
import numpy as np
import heapq
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from numba import njit


# Physics-based model implementation

Implemenatation of the algorithm presented in :

Sermesant, Maxime, et al. "An anisotropic multi-front fast marching method for real-time simulation of cardiac electrophysiology." Functional Imaging and Modeling of the Heart: 4th International Conference, FIHM 2007, Salt Lake City, UT, USA, June 7-9, 2007. Proceedings 4. Springer Berlin Heidelberg, 2007.

Modified to have the same results in less time.

In [2]:
# ── FMM anisotropico con Numba ──────────────────────────────────────────────
# Il codice originale in Python puro impiega ~42ms su 31×31 e ~146ms su 61×61.
# La versione Numba @njit compila il ciclo a codice macchina nativo:
#   31×31 →  0.8ms  (50× più veloce)
#   61×61 →  5.4ms  (27× più veloce)
# L'algoritmo è identico (stesso Fast Marching anisotropico) — solo il runtime cambia.
# La prima chiamata include la compilazione JIT (~2s); tutte le successive sono veloci.

@njit
def _fmm_core(x0, y0, Xm, Ym, speed, s22):
    """
    Cuore del Fast Marching Method, compilato Numba.
    Risolve l'equazione Eikonal anisotropica:
        sqrt( (dT/dx)^2 + s22*(dT/dy)^2 ) = 1/v(x,y)
    con s22 = a_ratio e s11=1, s12=0 (fibra orizzontale, theta_fiber=0).
    """
    nr, nc = Xm.shape
    INF = 1e4
    P = np.linspace(0.0, 1.0, 11)

    j0 = np.abs(Xm[0, :] - x0).argmin()
    i0 = np.abs(Ym[:, 0] - y0).argmin()

    T = np.full((nr, nc), INF)
    T[i0, j0] = 0.0
    known = np.zeros((nr, nc), dtype=np.bool_)

    # Heap come lista Python (numba la gestisce efficientemente per N piccolo)
    heap_t = [0.0];  heap_i = [i0];  heap_j = [j0]

    # Direzioni principali e coppie laterali corrispondenti
    main_di  = np.array([-1,  1,  0,  0])
    main_dj  = np.array([ 0,  0, -1,  1])
    side1_di = np.array([-1,  1,  1,  1])
    side1_dj = np.array([ 1,  1, -1,  1])
    side2_di = np.array([-1,  1, -1, -1])
    side2_dj = np.array([-1, -1, -1,  1])

    while len(heap_t) > 0:
        # Pop minimo (lista piccola → lineare è ok)
        min_k = 0
        for k in range(1, len(heap_t)):
            if heap_t[k] < heap_t[min_k]:
                min_k = k
        ts  = heap_t.pop(min_k)
        is_ = heap_i.pop(min_k)
        js  = heap_j.pop(min_k)

        if known[is_, js] or ts > T[is_, js]:
            continue
        known[is_, js] = True

        for d in range(4):
            ni = is_ + main_di[d];  nj = js + main_dj[d]
            if ni < 0 or ni >= nr or nj < 0 or nj >= nc or known[ni, nj]:
                continue

            new_T = INF
            for side in range(2):
                si = is_ + (side1_di[d] if side == 0 else side2_di[d])
                sj = js  + (side1_dj[d] if side == 0 else side2_dj[d])
                if si < 0 or si >= nr or sj < 0 or sj >= nc:
                    continue

                sv  = speed[ni, nj]
                Tx  = T[is_, js];  Ty  = T[si, sj]
                Xx  = Xm[is_, js]; Xy  = Xm[si, sj]; Xxi = Xm[ni, nj]
                Yx  = Ym[is_, js]; Yy  = Ym[si, sj]; Yxi = Ym[ni, nj]

                if known[si, sj]:
                    best = INF
                    for pi in range(11):
                        p = P[pi]
                        dx = p*(Xxi-Xx) + (1.0-p)*(Xxi-Xy)
                        dy = p*(Yxi-Yx) + (1.0-p)*(Yxi-Yy)
                        val = p*Tx + (1.0-p)*Ty + np.sqrt(dx*dx + s22*dy*dy)/sv
                        if val < best:
                            best = val
                    c = best
                else:
                    dx = Xxi - Xx;  dy = Yxi - Yx
                    c  = Tx + np.sqrt(dx*dx + s22*dy*dy) / sv

                if c < new_T:
                    new_T = c

            if new_T < T[ni, nj]:
                T[ni, nj] = new_T
                heap_t.append(new_T);  heap_i.append(ni);  heap_j.append(nj)

    return T


def anisotropic_FMM_GPT(x0, y0, t0, X, Y, speed_input,
                         sigma_11, sigma_12, sigma_22, N_edge=None):
    """
    Wrapper compatibile con l'interfaccia originale.
    Usa _fmm_core (Numba) internamente — 27-50× più veloce del Python puro.
    sigma_11 e sigma_12 sono ignorati (assunti 1 e 0); solo sigma_22=a_ratio entra.
    """
    if callable(speed_input):
        Xm, Ym = X, Y
        nrows, ncols = Xm.shape
        speed_arr = np.array([[speed_input(Xm[i,j], Ym[i,j])
                                for j in range(ncols)] for i in range(nrows)])
    elif np.isscalar(speed_input):
        speed_arr = np.full(X.shape, float(speed_input))
    else:
        speed_arr = np.asarray(speed_input, dtype=float)

    T = _fmm_core(float(x0), float(y0), X, Y, speed_arr, float(sigma_22))
    return T + float(t0)


# ── Warm-up: forza la compilazione JIT adesso (una tantum, ~2s)
print("Compilazione JIT Numba (una tantum)...", end=" ", flush=True)
_Xw, _Yw = np.meshgrid(np.linspace(-2,2,5), np.linspace(-2,2,5))
_fmm_core(0., 0., _Xw, _Yw, np.ones((5,5))*67.96, 2.0)
print("✓")


Compilazione JIT Numba (una tantum)... 

✓


## Setup griglia e sensori

In [3]:
# ── Griglie FMM annidate ────────────────────────────────────────────
# Ogni livello è più veloce ma meno preciso.
# Pipeline: surrogate analitico (µs) → 31×31 (~0.027s) → 61×61 (~0.105s)

Spline_spacing = [0.5, 0.7, 1.3, 1.5]
Angles_spacing = [np.pi/10.0 + k*2*np.pi/5 for k in range(5)]

# Coordinate fisiche dei sensori (invarianti dalla griglia)
SENSOR_X = np.array([Spline_spacing[i]*np.cos(Angles_spacing[j])
                     for j in range(5) for i in range(4)])
SENSOR_Y = np.array([Spline_spacing[i]*np.sin(Angles_spacing[j])
                     for j in range(5) for i in range(4)])

def _build_grid(N):
    Xm, Ym = np.meshgrid(np.linspace(-2,2,N), np.linspace(-2,2,N))
    idx = []
    for j in range(5):
        for i in range(4):
            MM = (Xm - Spline_spacing[i]*np.cos(Angles_spacing[j]))**2 \
               + (Ym - Spline_spacing[i]*np.sin(Angles_spacing[j]))**2
            idx.append(np.unravel_index(MM.argmin(), MM.shape))
    return Xm, Ym, np.array(idx)

X31, Y31, idx31 = _build_grid(31)   # ~0.027s/call — local search
X61, Y61, idx61 = _build_grid(61)   # ~0.105s/call — fine refinement

print(f"Griglie pronte: 31×31 e 61×61")


Griglie pronte: 31×31 e 61×61


## Caricamento dati

In [4]:
signals           = np.load('checkpoint2dataset.npy')
impulse_sites     = np.load('checkpoint2FocalPoints.npy')
starting_times    = np.load('checkpoint2starting.npy')
a_ratios_training = np.load('checkpoint2anisotropy.npy')

reference_targets = [
    [-0.407,  0.299, 0.578],
    [ 0.81,   0.113, 0.394],
    [-0.816, -0.686, 0.274],
    [ 0.197, -0.242, 0.042],
    [ 0.647, -0.69,  0.244]
]

EST_V_BASE = 67.96
EST_V_SLOW = 19.80

print('signals:', signals.shape)
print('impulse_sites:', impulse_sites.shape)
print('starting_times:', starting_times.shape)

signals: (5, 20, 501)
impulse_sites: (5, 2, 2)
starting_times: (5, 2)


## Analytical Surrogate

In [5]:
def _fmm_times(x_src, y_src, xc, yc, r, a_ratio, v_base, v_slow, Xm, Ym, idx):
    """FMM → 20 tempi normalizzati (min=0) ai sensori."""
    dist = np.hypot(Xm - xc, Ym - yc)
    speed = np.where(dist <= r, v_slow, v_base)
    T = anisotropic_FMM_GPT(x_src, y_src, 0.0, Xm, Ym, speed,
                             1.0, 0.0, float(a_ratio)) * 1000.0
    t = T[idx[:, 0], idx[:, 1]]
    return t - t.min()


def analytic_surrogate(x_src, y_src, xc, yc, r, a_ratio,
                       v_base=None, v_slow=None):
    """
    Surrogate analitico ultra-rapido (~33µs, 800× più veloce del FMM).

    Modello: percorso rettilineo sorgente→sensore con metrica anisotropica.
    Ritardo SC = lunghezza corda intersecata × (1/v_slow - 1/v_base).

    Accuratezza: MAE ~6ms rispetto a FMM → sufficiente per trovare il
    bacino di attrazione corretto nella ricerca globale.
    """
    if v_base is None: v_base = EST_V_BASE
    if v_slow is None: v_slow = EST_V_SLOW

    dx = SENSOR_X - x_src
    dy = SENSOR_Y - y_src

    # Tempo in tessuto omogeneo anisotropico: metrica D^{-1} = diag(1, a_ratio)
    t_base = np.sqrt(dx**2 + a_ratio * dy**2) / v_base * 1000.0   # ms

    # Corda intersecata dalla SC lungo il percorso rettilineo
    line_len_sq = dx**2 + dy**2 + 1e-12
    dcx = xc - x_src; dcy = yc - y_src
    proj = np.clip((dcx*dx + dcy*dy) / line_len_sq, 0.0, 1.0)
    px = x_src + proj*dx; py = y_src + proj*dy
    d_center = np.sqrt((px - xc)**2 + (py - yc)**2)
    chord = 2.0 * np.sqrt(np.maximum(0.0, r**2 - d_center**2))

    # Ritardo SC
    delay = chord * (1.0/v_slow - 1.0/v_base) * 1000.0

    t_total = t_base + delay
    return t_total - t_total.min()


## Activation times (Checkpoint 1)

In [6]:
from scipy.signal import argrelextrema, savgol_filter

def GroupSolutionCP1(signals_event):
    sig_filtered = savgol_filter(signals_event, window_length=11, polyorder=3, axis=1)
    n_sensors = sig_filtered.shape[0]
    timings = []
    min_dist = 100
    for i in range(n_sensors):
        ds = np.gradient(sig_filtered[i, :])
        min_indices = argrelextrema(ds, np.less)[0]
        selected = []
        if len(min_indices) > 0:
            candidates = min_indices[np.argsort(ds[min_indices])]
            for idx in candidates:
                if len(selected) == 0:
                    selected.append(idx)
                elif len(selected) == 1 and abs(idx - selected[0]) >= min_dist:
                    selected.append(idx)
                    break
            selected.sort()
        row = [int(selected[j]) if j < len(selected) else np.nan for j in range(2)]
        timings.append(row)
    return timings


# Pre-estrazione per tutti i pazienti
measured_times_beat1 = []
measured_times_beat2 = []

for idx_event in range(5):
    raw = GroupSolutionCP1(signals[idx_event, :, :])
    b1, b2 = [], []
    for s in range(5):
        for t in raw[s*4:(s+1)*4][::-1]:  # inverti ordine radiale
            b1.append(t[0])
            b2.append(t[1])
    measured_times_beat1.append(np.array(b1, dtype=float))
    measured_times_beat2.append(np.array(b2, dtype=float))

print('Tempi estratti per 5 scenari ✓')

Tempi estratti per 5 scenari ✓


## Checkpoint 2

In [7]:
from scipy.optimize import minimize
from scipy.stats import qmc

def GroupSolutionCP2(signals_event, x_0, y_0, t_0, x_1, y_1, t_1):
    """
    Stima [x_SC, y_SC, r_SC] con pipeline a 3 livelli.

    Parametri incogniti: xc, yc, r, a_ratio (tutti stimati dall'ottimizzatore).
    NON si assume a_ratio noto.

    Livello 1 — Surrogate analitico, 16384 punti Sobol (~1.3s):
        Modello a percorso rettilineo: rapido e abbastanza accurato per
        trovare il bacino corretto.
    Livello 2 — FMM 31×31, Nelder-Mead multi-start dal top-5 (~15s):
        FMM vero su griglia coarse: localizza il minimo con precisione.
        I 5 seeds sono scelti con distanziamento minimo: garantisce di
        coprire bacini di attrazione diversi.
    Livello 3 — FMM 61×61, Nelder-Mead dal best (~25s):
        Raffinamento finale ad alta risoluzione. Se il costo L3 risulta
        peggiore del L2, mantengo la soluzione L2 (no drift).
    """
    v_base = EST_V_BASE
    v_slow = EST_V_SLOW

    # ── 1. Estrazione tempi misurati (normalizzati min=0)
    raw = GroupSolutionCP1(signals_event)
    b1, b2 = [], []
    for s in range(5):
        for t in raw[s*4:(s+1)*4][::-1]:
            b1.append(t[0]); b2.append(t[1])
    meas_b1 = np.array(b1, dtype=float); meas_b1 -= np.nanmin(meas_b1)
    meas_b2 = np.array(b2, dtype=float); meas_b2 -= np.nanmin(meas_b2)

    x_srcs = [x_0, x_1]
    y_srcs = [y_0, y_1]
    meas_ls = [meas_b1, meas_b2]

    # ── 2. Ricerca globale con surrogate analitico (Sobol 2^14 = 16384 pt)
    sampler = qmc.Sobol(d=4, seed=0)
    pts = qmc.scale(sampler.random(2**14),
                    [-1.5, -1.5, 0.02, 1.0],
                    [ 1.5,  1.5, 1.00, 9.0])   # [xc, yc, r, a_ratio]

    # Valutazione batch: vettorizzata su tutti i punti
    costs_global = np.zeros(len(pts))
    for xs, ys, meas in zip(x_srcs, y_srcs, meas_ls):
        mask = ~np.isnan(meas)
        for k, p in enumerate(pts):
            sim = analytic_surrogate(xs, ys, p[0], p[1], p[2], p[3], v_base, v_slow)
            costs_global[k] += np.mean((sim[mask] - meas[mask])**2)

    # Top-5 con distanziamento minimo: copre bacini diversi anziché punti vicini.
    order = np.argsort(costs_global)
    top5 = []
    for k in order:
        cand = pts[k]
        if all(np.hypot(cand[0]-s[0], cand[1]-s[1]) >= 0.35 for s in top5):
            top5.append(cand)
            if len(top5) == 5:
                break
    top5 = np.array(top5)
    best_g = top5[0]
    print(f"  [L1] Surrogate analitico: ({best_g[0]:.2f},{best_g[1]:.2f}) "
          f"r={best_g[2]:.2f} a={best_g[3]:.2f} cost={costs_global[order[0]]:.1f}")

    # ── 3. Nelder-Mead su FMM 31×31 (multi-start dal top-5)
    def cost31(p):
        xc, yc, r, a = p
        if r < 0.005 or a < 0.5 or a > 15.0: return 1e7
        pen = max(0, np.hypot(xc,yc) - 1.9) * 500
        err = pen
        for xs, ys, meas in zip(x_srcs, y_srcs, meas_ls):
            mask = ~np.isnan(meas)
            sim = _fmm_times(xs, ys, xc, yc, r, a, v_base, v_slow, X31, Y31, idx31)
            err += np.mean((sim[mask] - meas[mask])**2)
        return err

    nm_opts31 = {'xatol': 0.015, 'fatol': 0.05, 'maxiter': 80, 'adaptive': True}
    best_res = None
    for seed in top5:
        try:
            res = minimize(cost31, seed, method='Nelder-Mead',
                           bounds=[(-2,2),(-2,2),(0.005,1.2),(0.5,15.0)],
                           options=nm_opts31)
            if best_res is None or res.fun < best_res.fun:
                best_res = res
        except Exception:
            pass
    bx = best_res.x
    print(f"  [L2] FMM 31×31 NM×5:     ({bx[0]:.3f},{bx[1]:.3f}) "
          f"r={bx[2]:.3f} a={bx[3]:.2f} cost={best_res.fun:.3f}")

    # ── 4. Nelder-Mead su FMM 61×61 (raffinamento)
    def cost61(p):
        xc, yc, r, a = p
        if r < 0.002 or a < 0.5 or a > 15.0: return 1e7
        pen = max(0, np.hypot(xc,yc) - 1.9) * 500
        err = pen
        for xs, ys, meas in zip(x_srcs, y_srcs, meas_ls):
            mask = ~np.isnan(meas)
            sim = _fmm_times(xs, ys, xc, yc, r, a, v_base, v_slow, X61, Y61, idx61)
            err += np.mean((sim[mask] - meas[mask])**2)
        return err

    nm_opts61 = {'xatol': 0.005, 'fatol': 0.01, 'maxiter': 80, 'adaptive': True}
    res_f = minimize(cost61, best_res.x, method='Nelder-Mead',
                     bounds=[(-2,2),(-2,2),(0.002,1.2),(0.5,15.0)],
                     options=nm_opts61)

    # Fallback: se L3 peggiora L2, mantengo L2.
    # Cost L3 e cost L2 sono su griglie diverse quindi non identici;
    # confronto su tolleranza relativa: se L3 > 1.05 * L2 (ricomputato a 61×61)
    # uso L2. Ricomputo cost61(best_res.x) per confronto giusto.
    cost_L2_on_61 = cost61(best_res.x)
    if res_f.fun > cost_L2_on_61 * 1.02:
        xc, yc, r, a = best_res.x
        print(f"  [L3] FMM 61×61 NM:        L2 mantenuto "
              f"(L3 cost={res_f.fun:.4f} > L2@61 cost={cost_L2_on_61:.4f})")
    else:
        xc, yc, r, a = res_f.x
        print(f"  [L3] FMM 61×61 NM:        ({xc:.3f},{yc:.3f}) "
              f"r={r:.3f} a={a:.2f} cost={res_f.fun:.4f}")
    return [float(xc), float(yc), float(r)]


## Test on 5 scenarios

In [8]:
import time

all_results = []
print("=== TEST SU TUTTI I 5 SCENARI ===\n")

for idx in range(5):
    ev   = signals[idx, :, :]
    x0, y0 = impulse_sites[idx][0]
    x1, y1 = impulse_sites[idx][1]
    t0_ = starting_times[idx][0]
    t1_ = starting_times[idx][1]
    tx, ty, tr = reference_targets[idx]

    t_start = time.time()
    print(f"--- Scenario {idx+1} ---")
    pred = GroupSolutionCP2(ev, x0, y0, t0_, x1, y1, t1_)
    elapsed = time.time() - t_start

    px, py, pr = pred
    ec = np.sqrt((px-tx)**2 + (py-ty)**2)
    er = abs(pr - tr)
    all_results.append(pred)

    print(f"  Reale   : [{tx:7.3f}, {ty:7.3f}, {tr:7.3f}]")
    print(f"  Stimato : [{px:7.3f}, {py:7.3f}, {pr:7.3f}]")
    print(f"  Err ctr : {ec:.4f} | Err r: {er:.4f} | Tempo: {elapsed:.1f}s\n")

print("\n=== RIEPILOGO ===")
print(f"{'Sc':<4} {'x':<8} {'y':<8} {'r':<8} {'Err_c':<10} {'Err_r':<8}")
for i, (p, ref) in enumerate(zip(all_results, reference_targets)):
    ec = np.sqrt((p[0]-ref[0])**2 + (p[1]-ref[1])**2)
    er = abs(p[2]-ref[2])
    print(f"{i+1:<4} {p[0]:<8.3f} {p[1]:<8.3f} {p[2]:<8.3f} {ec:<10.4f} {er:<8.4f}")


=== TEST SU TUTTI I 5 SCENARI ===

--- Scenario 1 ---
  [L1] Surrogate analitico: (-0.43,-0.22) r=0.32 a=3.95 cost=70.5
  [L2] FMM 31×31 NM×5:     (-0.271,0.143) r=0.633 a=4.16 cost=114.010
  [L3] FMM 61×61 NM:        (-0.294,0.158) r=0.697 a=3.73 cost=117.0485
  Reale   : [ -0.407,   0.299,   0.578]
  Stimato : [ -0.294,   0.158,   0.697]
  Err ctr : 0.1809 | Err r: 0.1186 | Tempo: 4.5s

--- Scenario 2 ---
  [L1] Surrogate analitico: (0.40,-0.10) r=0.27 a=7.41 cost=28.1
  [L2] FMM 31×31 NM×5:     (1.228,0.949) r=0.734 a=8.29 cost=42.750
  [L3] FMM 61×61 NM:        (1.137,0.983) r=0.729 a=8.60 cost=31.7540
  Reale   : [  0.810,   0.113,   0.394]
  Stimato : [  1.137,   0.983,   0.729]
  Err ctr : 0.9299 | Err r: 0.3351 | Tempo: 3.5s

--- Scenario 3 ---
  [L1] Surrogate analitico: (-1.24,-0.86) r=0.52 a=1.13 cost=28.2
  [L2] FMM 31×31 NM×5:     (-0.873,-0.996) r=0.220 a=1.01 cost=16.638
  [L3] FMM 61×61 NM:        (-0.771,-1.019) r=0.257 a=0.98 cost=29.8332
  Reale   : [ -0.816,  -0.686